In [0]:
%sql
SELECT u.usage_date,
       ROUND(SUM(u.usage_quantity * p.pricing.default), 2) AS cost_usd
FROM system.billing.usage u
JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.usage_start_time >= p.price_start_time
 AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
WHERE u.usage_date >= current_date() - 14
GROUP BY u.usage_date
ORDER BY u.usage_date

In [0]:
%sql
SELECT
  coalesce(u.usage_metadata.notebook_path, u.usage_metadata.job_id, u.billing_origin_product) AS source,
  ROUND(SUM(u.usage_quantity), 2) AS dbus,
  ROUND(SUM(u.usage_quantity * p.pricing.default), 2) AS cost_usd,
  MAX(u.usage_end_time) AS latest_usage_recorded
FROM system.billing.usage u
JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.usage_start_time >= p.price_start_time
 AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
WHERE u.usage_date = current_date()
GROUP BY ALL
ORDER BY cost_usd DESC